# 1 · Track my hand clips and replay them on the robot (CPU runtime)

Phone video → hand path on the table (cm) → robot path → replay in LIBERO → successful replays saved as SmolVLA training demos.

This notebook only calls the scripts in the repo; all logic lives in `src/hand2panda/`.
Run the cells top to bottom. A CPU runtime is enough.

Layouts 0–24 are reserved for testing SmolVLA (notebook 2), so replays here use layouts 25–49.

## 0 · Setup (≈3 min, once per session)

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
import os
if not os.path.exists("/content/repo"):
    !git clone -q https://github.com/Suhail-Halaby/humanoidRLassignment.git /content/repo
if not os.path.exists("/content/LIBERO"):
    !git clone -q https://github.com/Lifelong-Robot-Learning/LIBERO.git /content/LIBERO
!pip install -q -r /content/repo/requirements-sim.txt -r /content/repo/requirements-track.txt
!pip install -q -e /content/LIBERO
!pip install -q -e /content/repo
!cd /content/repo && git pull -q && git log --oneline -1

## 1 · Settings
Examiners: leave `MODE = "examiner"`. Change the rest only if you want to, then run the cells below. `CLIP` is the single clip used for the quick check (sections 2–3).

In [ ]:
# Who is running this notebook?
#   "examiner": read my shared Drive folder (added to your Drive as a shortcut called humanoid,
#               see README "Data access"); everything you produce is written to /content/out
#   "author":   my own Drive folder, read and write
MODE = "examiner"
if MODE == "author":
    DATA = OUT = "/content/drive/MyDrive/humanoid"
else:
    DATA, OUT = "/content/drive/MyDrive/humanoid", "/content/out"
    os.makedirs(OUT, exist_ok=True)
assert os.path.exists(DATA), f"{DATA} not found: add the shared folder to your Drive (README, Data access)"

VIDEOS = f"{DATA}/videos"           # the phone clips (input)
PROCESSED = f"{OUT}/processed_v2"   # tracking results (.npz, .png, overlay .mp4)
REPLAY = f"{OUT}/replay_v2"         # replay videos + results table
DEMOS = f"{OUT}/demos"              # successful replays saved as training demos

CLIP = "clip20"                   # clip for the quick check
COLOUR = "yellow"                 # object colour
MARKER_CM = 5.0                   # ruler-measured side of one printed marker
REPLAY_ARGS = "--settle 30 --lift 0.10 --clear 0.25 --release-above 0.20 --ramp 0.03"   # see README step 3-4
FIRST_CLIP = 20                   # section 4 uses clips FIRST_CLIP and up

## 2 · Track one clip
Checks to make in the output line `tracking:`: hand seen > ~90 %, object seen well above 0 %, and `close@… open@…` (not NO GRASP FOUND).
In the overlay video the blue ring should sit on the object and the dot should turn red while holding it.

In [ ]:
!rm -rf /content/one_clip && mkdir -p /content/one_clip /content/one_track
!cp {VIDEOS}/{CLIP}.* /content/one_clip/ && ls /content/one_clip
!python /content/repo/scripts/track.py --videos /content/one_clip --out {PROCESSED} --colour {COLOUR} --marker-cm {MARKER_CM}

In [ ]:
import mediapy
from IPython.display import Image, display
display(Image(f"{PROCESSED}/{CLIP}.png"))
mediapy.show_video(mediapy.read_video(f"{PROCESSED}/{CLIP}_overlay.mp4"), fps=15, width=360)

## 3 · Replay that clip on one layout
Prints `layout 25: SUCCESS / fail | …`. If it fails, watch the video and note the first step that goes wrong:
reach the can → close on it → lift → carry → open over the basket → can lands inside.

In [ ]:
!rm -rf /content/one_track && mkdir -p /content/one_track
!cp {PROCESSED}/{CLIP}.npz /content/one_track/
%cd /content/repo
!PYTHONPATH=/content/LIBERO python scripts/replay.py --tracks /content/one_track --out {REPLAY} --inits 25 {REPLAY_ARGS}

In [ ]:
import mediapy
mediapy.show_video(mediapy.read_video(f"{REPLAY}/{CLIP}_layout25.mp4"), fps=20, width=360)

## 4 · All clips: track, then record training demos
Only run this once section 3 succeeds. Tracking all clips ≈ 1 min per clip; recording ≈ a few seconds per replay
(clips × 5 layouts). Every successful replay is saved to `DEMOS` as one `.npz` training demo.

In [ ]:
# only the re-filmed clips (20 and up): clips 1-19 did not match the task
import glob, os, re, shutil
NEW = "/content/new_clips"
shutil.rmtree(NEW, ignore_errors=True); os.makedirs(NEW)
for f in sorted(glob.glob(f"{VIDEOS}/clip*.*")):
    if int(re.search(r"clip(\d+)", os.path.basename(f)).group(1)) >= FIRST_CLIP:
        shutil.copy(f, NEW)
print(sorted(os.listdir(NEW)))
!python /content/repo/scripts/track.py --videos {NEW} --out {PROCESSED} --colour {COLOUR} --marker-cm {MARKER_CM} --no-video

In [ ]:
LAYOUTS = ",".join(str(i) for i in range(25, 30))     # 5 demo layouts per clip; 0-24 are the test set
%cd /content/repo
!PYTHONPATH=/content/LIBERO python scripts/replay.py --tracks {PROCESSED} --out {REPLAY} --inits {LAYOUTS} {REPLAY_ARGS} --record {DEMOS}
!ls {DEMOS} | wc -l

In [ ]:
# check: every demo file is a success from this run
import csv
rows = list(csv.DictReader(open(f"{REPLAY}/replay_results.csv")))
wins = {f"{r['clip']}_layout{r['layout']}.npz" for r in rows if r["result"] == "success"}
files = {f for f in os.listdir(DEMOS) if f.endswith(".npz")}
print(f"replays: {len(rows)}   successes: {len(wins)}   demo files: {len(files)}")
print("demo files that are NOT a success in this run:", sorted(files - wins) or "none")

Next: notebook 2 (GPU) builds the dataset from `DEMOS`, post-trains SmolVLA and benchmarks it.